In [7]:
from PandaTableVoxelClutterGenerator import PandaTableVoxelClutterGenerator
import robotic as ry
import numpy as np


def main():
    generator = PandaTableVoxelClutterGenerator(
        base_scene_file=ry.raiPath("../rai-robotModels/scenarios/pandaSingle.g"),  # Path to the base Panda scene file
        voxel_dir="../voxel_generation/data",                                        # Directory containing voxel .g files
        output_dir="./generated_envs",                                               # Directory where generated environments will be saved
        table_frame_name="table",                                                    # Name of the table frame inside the scene
        gap=0.04,                                                                    # Spacing margin used around objects / boundaries in placement logic
        spawn_height=0.41,                                                           # Extra height above the table used when initially dropping voxels
        seed=np.random.randint(0, 400),                                                                      # Random seed for reproducible scene generation
        per_cube_mass=0.2,                                                           # Mass assigned to each cube piece of a voxel object
        table_shape_size=(1.6, 1.6, 0.08, 0.02),                                    # Table size/shape parameters passed to the table ssBox
        panda_base_relative_pos=(0.0, 0.0, 0.05),                                   # Relative position offset applied to the Panda base
        target_alpha=0.9,                                                            # Alpha/transparency value used for the target voxel cubes
        target_center_jitter_ratio=0.05,                                             # Small random jitter ratio around the center of the target quarter
        clutter_mode="low_clutter",                                                 # Clutter placement mode: "random", "low_clutter", or "high_clutter"
        placement_candidate_count=128,                                               # Number of sampled candidates used mainly in low/high clutter placement
    )

    C, summary = generator.create_environment_with_refill(
        num_voxels=8,                # Total number of voxels in the final scene, including the target
        sim_seconds=10.0,            # Physics simulation duration per round in seconds
        sim_dt=0.01,                 # Physics simulation timestep
        max_refill_rounds=10,        # Maximum number of clutter refill/simulate rounds
        xy_margin=0.02,              # XY table boundary margin used when checking whether objects stayed on the table
        z_tolerance=0.15,            # Allowed drop below table-top threshold before object is considered off-table
        batch_spawn_count=5,         # Maximum number of clutter voxels spawned in one refill batch
    )

    print("\n--- Scene summary ---")
    print(f"Clutter mode             : {summary['clutter_mode']}")
    print(f"Final on table           : {summary['final_on_table']}")
    print(f"Final off table          : {summary['final_off_table']}")
    print(f"Refill rounds            : {summary['rounds']}")

    print("\n--- Target voxel info ---")
    print(f"Target voxel file        : {summary['target_voxel_file']}")
    print(f"Target voxel basename    : {summary['target_voxel_basename']}")
    print(f"Original voxel basename  : {summary['target_original_voxel_basename']}")
    print(f"Target quarter           : {summary['target_quarter_mode']}")
    print(f"Target object prefix     : {summary['target_object_prefix']}")
    print(f"Target alpha             : {summary['target_alpha']}")

    alive_objects = [obj for obj in summary["objects"] if obj["alive"]]
    print(f"\nAlive spawned objects    : {len(alive_objects)}")
    print("Alive voxel basenames    :", [obj["basename"] for obj in alive_objects])

    target_alive = [obj for obj in alive_objects if obj.get("is_target", False)]
    clutter_alive = [obj for obj in alive_objects if not obj.get("is_target", False)]

    print(f"\nAlive target objects     : {len(target_alive)}")
    print(f"Alive clutter objects    : {len(clutter_alive)}")

    if target_alive:
        tgt = target_alive[0]
        print("\n--- Alive target object details ---")
        print(f"Prefix                   : {tgt['prefix']}")
        print(f"Basename                 : {tgt['basename']}")
        print(f"Original basename        : {tgt['original_basename']}")
        print(f"Spawn XY                 : {tgt['spawn_xy']}")
        print(f"Spawn Z                  : {tgt['spawn_z']}")
        print(f"Theta deg                : {tgt['theta_deg']}")
        print(f"Region modes             : {tgt['region_modes']}")
        print(f"Alpha                    : {tgt['target_alpha']}")

    if clutter_alive:
        print("\n--- Alive clutter object details ---")
        for i, obj in enumerate(clutter_alive, start=1):
            print(f"\nClutter #{i}")
            print(f"  Prefix                 : {obj['prefix']}")
            print(f"  Basename               : {obj['basename']}")
            print(f"  Original basename      : {obj['original_basename']}")
            print(f"  Spawn XY               : {obj['spawn_xy']}")
            print(f"  Spawn Z                : {obj['spawn_z']}")
            print(f"  Theta deg              : {obj['theta_deg']}")
            print(f"  Region modes           : {obj['region_modes']}")
            print(f"  Clutter mode           : {obj.get('clutter_mode')}")

    saved_path = generator.save_environment(
        C,
        file_name="generated_panda_table_voxel_clutter.g",  # Output filename for the generated environment
    )
    print("\nSaved to:", saved_path)

    # C.view(True)  # Uncomment to visualize the generated scene


if __name__ == "__main__":
    main()

Chosen target voxel: 726.g
Chosen target quarter: front_right
Clutter mode: low_clutter
Initially spawned extra voxels: 5 / 5

=== Clutter simulation round 1 ===
Clutter on table: 5 | Off table: 0 | Clutter target: 7
Trying to respawn up to 2 clutter voxel(s)...
Respawned 2 clutter voxel(s).

=== Clutter simulation round 2 ===
Clutter on table: 7 | Off table: 0 | Clutter target: 7
Desired number of clutter voxels is on the table.

=== Spawning target last ===

=== Target spawn diagnostics ===
Target voxel file         : ../voxel_generation/data/726.g
Target quarter            : front_right
Target quarter bounds     : x[0.0000, 0.8000] y[-0.8000, 0.0000]
On-table bounds (margin)  : x[-0.7800, 0.7800] y[-0.7800, 0.7800]
Table top z               : 0.6400
z_tolerance               : 0.1500
Survival threshold z      : 0.4900
sim_seconds               : 10.0
sim_dt                    : 0.01
max_spawn_attempts        : 40

--- Target attempt 1/40 ---
Spawn prefix              : goal_obj7_
Sp

In [8]:
# The generated environment is saved to generated_envs
C = ry.Config()
C.addFile("generated_envs/generated_panda_table_voxel_clutter.g") 

C.view()


0